# Analisando a tabela curada

Por simplicidade, usarei somente Python e o DuckDB para as análises. Como se trata de um volume de dados pequeno, não há necessidade de distribuir o processamento usando Apache Spark. Claro que, em um ambiente produtivo, isso pode ser necessário, mas por não ter tais requisitos na requisição da demanda, optei por adotar a estratégia que atende à celeridade das análises.

Outro ponto ao considerar o DuckDB em conjunto com SQL ao invés de Python, é facilitar a análise por outras áreas que usam o SQL como meio principal para leitura dos dados, abrindo o escopo de interpretação do que foi feito para as áreas de negócio.

## Instalando e importando bibliotecas necessárias

In [2]:
!pip install deltalake python-dotenv duckdb

In [3]:
import os

import duckdb
from deltalake import DeltaTable
from dotenv import load_dotenv

load_dotenv()

False

## Configurando o ambiente e instanciando os serviços

In [4]:
storage = {
    "AWS_ENDPOINT_URL": f"http://{os.environ['MINIO_ENDPOINT']}",
    "AWS_ACCESS_KEY_ID": os.environ["MINIO_ACCESS_KEY"],
    "AWS_SECRET_ACCESS_KEY": os.environ["MINIO_SECRET_KEY"],
    "AWS_REGION": "us-east-1",
    "AWS_ALLOW_HTTP": "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}

table = DeltaTable("s3://gold/ans_data", storage_options=storage)
df = table.to_pandas()
conn = duckdb.connect()
conn.register("gold_df", df)

,DT_CARGA,count_star()
0,2026-05-20,84175


## Analisando a composição da tabela

In [ ]:
conn.execute("SELECT DT_CARGA, COUNT(*) FROM gold_df GROUP BY DT_CARGA").df()

In [ ]:
conn.execute("DESCRIBE gold_df").df()

In [21]:
conn.execute("SUMMARIZE gold_df").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,ID_CMPT_MOVEL,VARCHAR,2025-08,2025-08,1,NaN,NaN,NaN,NaN,NaN,84175,0.0
1,CD_OPERADORA,BIGINT,582,424188,402,293037.79133947135,121776.51429700614,301949,313083,358399,84175,0.0
2,NM_RAZAO_SOCIAL,VARCHAR,2CARE OPERADORA DE SAÚDE LTDA.,W. DENTAL PLANOS ODONTOLÓGICOS S.A,332,NaN,NaN,NaN,NaN,NaN,84175,0.0
3,NR_CNPJ,BIGINT,6037000127,97388490000187,374,36823605523352.7,29090104629737.145,4313123233583,37313475000148,58119199000151,84175,0.0
4,MODALIDADE_OPERADORA,VARCHAR,AUTOGESTÃO,SEGURADORA ESPECIALIZADA EM SAÚDE,7,NaN,NaN,NaN,NaN,NaN,84175,0.0
5,SG_UF,VARCHAR,TO,TO,1,NaN,NaN,NaN,NaN,NaN,84175,0.0
6,CD_MUNICIPIO,BIGINT,170000,172210,144,171300.44482328484,758.5865924917722,170550,171560,172100,84175,0.0
7,NM_MUNICIPIO,VARCHAR,Abreulândia,Xambioá,176,NaN,NaN,NaN,NaN,NaN,84175,0.0
8,TP_SEXO,VARCHAR,F,M,2,NaN,NaN,NaN,NaN,NaN,84175,0.0
9,DE_FAIXA_ETARIA,VARCHAR,1 a 4 anos,Menos de 1 ano,17,NaN,NaN,NaN,NaN,NaN,84175,0.0


## Consultas solicitadas

### a) Quais são as **5 operadoras** com **maior número de beneficiários ativos**?

In [11]:
conn.execute("""
    SELECT 
        CD_OPERADORA,
        NM_RAZAO_SOCIAL,
        CAST(SUM(QT_BENEFICIARIO_ATIVO) AS INT) AS QT_TOTAL_BENEFICIARIOS_ATIVOS
    FROM gold_df
    GROUP BY 1, 2 ORDER BY 3 DESC
    LIMIT 5
""").df()

,CD_OPERADORA,NM_RAZAO_SOCIAL,QT_TOTAL_BENEFICIARIOS
0,374440,PREVIDENT ASSISTÊNCIA ODONTOLÓGICA S.A,67430
1,301949,ODONTOPREV S/A,52832
2,309907,UNIMED PALMAS COOPERATIVA DE TRABALHO MÉDICO,29356
3,5711,BRADESCO SAÚDE S.A.,16280
4,313084,COOPERATIVA DE TRABALHO MEDICO DE ARAGUAÍNA - ...,14355


### b) Qual é a **faixa etária com mais beneficiários** e **quantos** são?

A pergunta ficou um pouco ambígua com relação à qualificação dos beneficiários, podendo ser: ativos OU a soma de ativos e aderidos. Seguem abaixo as análises em ambos os casos:

#### b1) Considerando somente beneficiários ativos:

In [13]:
conn.execute("""
    SELECT
        DE_FAIXA_ETARIA, 
        DE_FAIXA_ETARIA_REAJ, 
        CAST(SUM(QT_BENEFICIARIO_ATIVO) AS INT) AS QT_TOTAL_BENEFICIARIOS_ATIVOS
    FROM gold_df
    GROUP BY 1, 2 ORDER BY 3 DESC
    LIMIT 1
""").df()

,DE_FAIXA_ETARIA,DE_FAIXA_ETARIA_REAJ,QT_TOTAL_BENEFICIARIOS_ATIVOS
0,35 a 39 anos,34 a 38 anos,21967


#### b2) Considerando a soma de ativos e aderidos:

In [17]:
conn.execute("""
    SELECT
        DE_FAIXA_ETARIA, 
        DE_FAIXA_ETARIA_REAJ, 
        CAST(SUM(QT_BENEFICIARIO_ATIVO) AS INT)
        + CAST(SUM(QT_BENEFICIARIO_ADERIDO) AS INT) AS QT_TOTAL_BENEFICIARIOS
    FROM gold_df
    GROUP BY 1, 2 ORDER BY 3 DESC
    LIMIT 1
""").df()

,DE_FAIXA_ETARIA,DE_FAIXA_ETARIA_REAJ,QT_TOTAL_BENEFICIARIOS
0,35 a 39 anos,34 a 38 anos,22434


### c) Liste, de forma **decrescente**, a **quantidade de beneficiários por município**.

Seguindo o mesmo problema de ambiguidade da questão acima, podemos ter a quantidade total considerando somente ativos OU a soma de ativos e aderidos aos planos.

#### c1) Considerando somente beneficiários ativos:

In [19]:
conn.execute("""
    SELECT
        NM_MUNICIPIO,
        SG_UF,
        CAST(SUM(QT_BENEFICIARIO_ATIVO) AS INT) AS QT_TOTAL_BENEFICIARIOS_ATIVOS
    FROM gold_df
    GROUP BY 1, 2 ORDER BY 3 DESC
""").df()

,NM_MUNICIPIO,SG_UF,QT_TOTAL_BENEFICIARIOS_ATIVOS
0,Palmas,TO,127812
1,Araguaína,TO,37846
2,Gurupi,TO,18251
3,Porto Nacional,TO,15821
4,Paraíso do Tocantins,TO,7542
...,...,...,...
135,Ipueiras,TO,38
136,Monte Santo do Tocantins,TO,37
137,Santa Rita do Tocantins,TO,35
138,Chapada de Areia,TO,26


#### c2) Considerando a soma de ativos e aderidos:

In [20]:
conn.execute("""
    SELECT
        NM_MUNICIPIO,
        SG_UF,
        CAST(SUM(QT_BENEFICIARIO_ATIVO) AS INT)
        + CAST(SUM(QT_BENEFICIARIO_ADERIDO) AS INT) AS QT_TOTAL_BENEFICIARIOS
    FROM gold_df
    GROUP BY 1, 2 ORDER BY 3 DESC
""").df()

,NM_MUNICIPIO,SG_UF,QT_TOTAL_BENEFICIARIOS
0,Palmas,TO,130810
1,Araguaína,TO,38682
2,Gurupi,TO,18604
3,Porto Nacional,TO,16340
4,Paraíso do Tocantins,TO,7721
...,...,...,...
135,Monte Santo do Tocantins,TO,38
136,Ipueiras,TO,38
137,Santa Rita do Tocantins,TO,35
138,Chapada de Areia,TO,27
